#Andmete laadimine uuesti

In [2]:
import pandas as pd
import os
from dotenv import load_dotenv
from supabase import create_client

In [3]:
load_dotenv()

supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

In [4]:
def get_data(table_name):
    data = []
    page_size = 1000
    page = 0

    while True:
        response = supabase.table(table_name).select("*").range(
            page * page_size,
            (page + 1) * page_size - 1
        ).execute()

        data.extend(response.data)

        if len(response.data) < page_size:
            break

        page += 1

    return pd.DataFrame(data)

In [5]:
df_sales = get_data("sales")
df_customers = get_data("customers")

df = pd.merge(
    df_sales,
    df_customers,
    on="customer_id",
    how="left"
)

df.shape

(10118, 20)

#Duplikaatide kontroll

Kontrollin, kas andmestikus esineb korduvaid ridu.
Duplikaadid võivad põhjustada valesid analüüsitulemusi, sest sama müük või klient arvestatakse mitu korda.

In [6]:
print("Algne kuju:", df.shape)

Algne kuju: (10118, 20)


In [7]:
print("Duplikaadid:", df.duplicated().sum())

Duplikaadid: 0


#Eemaldan duplikaadid, tegelikult neid ei esinenud. 

In [8]:
df = df.drop_duplicates()

In [9]:
print("Pärast eemaldamist:", df.shape)

Pärast eemaldamist: (10118, 20)


#Puuduvate väärtuste kontroll

Kontrollin kõikide veergude puuduvate väärtuste arvu.
Kriitilised väljad (customer_id, sale_date, total_price) peavad olema täidetud.

In [10]:
print(df.isnull().sum())

id                      0
sale_id                 0
invoice_id              0
sale_date               0
customer_id           988
product_id              0
quantity                0
unit_price              0
total_price             0
channel                 0
store_location       3462
payment_method          0
first_name            988
last_name             988
email                1944
phone                 988
city                  988
registration_date     988
loyalty_tier         4660
birth_year            988
dtype: int64


#Kriitiliste puuduvate väärtuste eemaldamine

Eemaldan read, kus puudub oluline müügiinfo.
See tagab, et hilisem RFM analüüs põhineb täielikel andmetel.

In [11]:
df = df.dropna(
    subset=["customer_id", "sale_date", "total_price"]
)

df.shape

(9130, 20)

#Kuupäeva formaadi muutmine

Teisendan sale_date väärtused datetime formaati.
See võimaldab arvutada ostude ajavahemikke ja viimase ostu kuupäeva.

In [12]:
df["sale_date"] = pd.to_datetime(df["sale_date"])

df["sale_date"].dtype

dtype('<M8[us]')

In [13]:
df["sale_date"].dtype

dtype('<M8[us]')

#Outlier'ite kontroll

Kontrollin negatiivseid total_price väärtusi.
Negatiivsed müügid eemaldan, sest need ei sobi kliendikulutuste analüüsi.

In [14]:
(df["total_price"] < 0).sum()

np.int64(180)

In [15]:
df = df[df["total_price"] > 0]

df.shape

(8950, 20)

#Puhastusraport

Koostan kokkuvõtte puhastatud andmetest:
- lõplik ridade arv
- unikaalsete klientide arv
- analüüsi kuupäevavahemik

In [16]:
print("Lõplik kuju:", df.shape)
print("Unikaalsed kliendid:", df["customer_id"].nunique())
print("Kuupäeva algus:", df["sale_date"].min())
print("Kuupäeva lõpp:", df["sale_date"].max())

Lõplik kuju: (8950, 20)
Unikaalsed kliendid: 2540
Kuupäeva algus: 2023-01-01 00:00:00
Kuupäeva lõpp: 2026-05-17 00:00:00


# Puhastusraport

- Algselt ridu: 10118
- Pärast puhastamist: 8950
- Eemaldatud duplikaate: 0
- Eemaldatud NULL customer_id ridu: 988
- Eemaldatud negatiivseid total_price ridu: 180
- Unikaalseid kliente: 2540
- Andmeperiood: 2023-01-01 kuni 2026-05-17

#Loon faili puhastatud andmete jaoks, et saaks järgmist ülesannet täita lihtsamalt. 

In [17]:
df.to_csv("clean_customers_sales.csv", index=False)